In [3]:
# Import Libraries
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from gensim import corpora
from gensim.models import LdaModel

# Download necessary NLTK Resources silently
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

# Initialize our text cleaning tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

print("Cell 1 Complete: All libraries loaded and ready!")

Cell 1 Complete: All libraries loaded and ready!


In [4]:
# Define the short documents
short_documents = [  
    "Rafael Nadal Joins Roger Federer in Missing U.S. Open",  
    "Rafael Nadal Is Out of the Australian Open",  
    "Biden Announces Virus Measures",  
    "Biden's Virus Plans Meet Reality",  
    "Where Biden's Virus Plan Stands"  
] 

# Create the preprocessing function
def preprocess_text(text):  
    tokens = word_tokenize(text.lower())  # Tokenize and lowercase
    tokens = [token for token in tokens if token.isalnum()]  # Keep only letters/numbers
    tokens = [token for token in tokens if token not in stop_words]  # Remove stopwords
    tokens = [lemmatizer.lemmatize(token) for token in tokens]  # Lemmatize
    return tokens  

# Apply preprocessing to our short documents
preprocessed_short_docs = [preprocess_text(doc) for doc in short_documents]

print("Cell 2 Complete: Short documents preprocessed!")
print(preprocessed_short_docs[0]) # Peek at the first one

Cell 2 Complete: Short documents preprocessed!
['rafael', 'nadal', 'join', 'roger', 'federer', 'missing', 'open']


In [5]:
# Create a document-term matrix
dictionary_short = corpora.Dictionary(preprocessed_short_docs)
corpus_short = [dictionary_short.doc2bow(doc) for doc in preprocessed_short_docs]

# Run LDA with 2 topics
lda_model_short = LdaModel(corpus_short, num_topics=2, id2word=dictionary_short, passes=15)

# Interpret Results
print("--- Example 1: Top terms for each topic ---")
for topic_id in range(lda_model_short.num_topics):
    print(f"Topic #{topic_id}:")
    top_terms = lda_model_short.show_topic(topic_id, topn=5)
    print([term[0] for term in top_terms])
    print()

--- Example 1: Top terms for each topic ---
Topic #0:
['virus', 'biden', 'open', 'nadal', 'rafael']

Topic #1:
['plan', 'biden', 'virus', 'rafael', 'nadal']



In [6]:
print("Loading NPR dataset...")
# Load the Data with error handling for the broken row
df = pd.read_csv('npr.csv', engine='python', on_bad_lines='skip')  
npr_documents = df['Article'].tolist()  

print(f"Loaded {len(npr_documents)} articles. Preprocessing now (this may take a minute)...")
# Preprocess each document in the list
preprocessed_npr_docs = [preprocess_text(doc) for doc in npr_documents]  

print("\nCell 4 Complete: NPR data preprocessed!")
print("Sample preprocessed tokens:", preprocessed_npr_docs[0][:10])

Loading NPR dataset...
Loaded 11992 articles. Preprocessing now (this may take a minute)...

Cell 4 Complete: NPR data preprocessed!
Sample preprocessed tokens: ['washington', '2016', 'even', 'policy', 'bipartisan', 'politics', 'sense', 'year', 'show', 'little']


In [7]:
# Create a Gensim Dictionary
dictionary_npr = corpora.Dictionary(preprocessed_npr_docs)   
# Filter out tokens appearing in less than 15 docs or more than 50% of docs
dictionary_npr.filter_extremes(no_below=15, no_above=0.5)  

# Convert to bag-of-words
corpus_npr = [dictionary_npr.doc2bow(doc) for doc in preprocessed_npr_docs]   

print("Running LDA Model on NPR data (finding 5 topics)...")
# Train LDA model
lda_model_npr = LdaModel(corpus_npr, num_topics=5, id2word=dictionary_npr, passes=15)

# Interpret Results: Match documents to their dominant topic
article_labels = []  
for doc in preprocessed_npr_docs:  
    bow = dictionary_npr.doc2bow(doc)  
    topics = lda_model_npr.get_document_topics(bow)  
    dominant_topic = max(topics, key=lambda x: x[1])[0]  
    article_labels.append(dominant_topic)  
 
# Create and print the DataFrame
df_result = pd.DataFrame({"Article": npr_documents, "Topic": article_labels})  
print("\n--- Table with Articles and Topic (First 5 Rows) ---")  
print(df_result.head())  
print("\n--- Top Terms for Each Topic with Weights ---")  
for idx, topic in lda_model_npr.print_topics():  
    print(f"Topic {idx}:")  
    terms = [term.strip() for term in topic.split("+")]  
    for term in terms:  
        weight, word = term.split("*")  
        print(f" - {word.strip()} (weight: {weight.strip()})")  
    print()

Running LDA Model on NPR data (finding 5 topics)...

--- Table with Articles and Topic (First 5 Rows) ---
                                             Article  Topic
0  In the Washington of 2016, even when the polic...      0
1    Donald Trump has used Twitter  —   his prefe...      0
2    Donald Trump is unabashedly praising Russian...      0
3  Updated at 2:50 p. m. ET, Russian President Vl...      0
4  From photography, illustration and video, to d...      4

--- Top Terms for Each Topic with Weights ---
Topic 0:
 - "trump" (weight: 0.021)
 - "president" (weight: 0.009)
 - "clinton" (weight: 0.008)
 - "state" (weight: 0.008)
 - "campaign" (weight: 0.006)
 - "republican" (weight: 0.005)
 - "election" (weight: 0.005)
 - "obama" (weight: 0.004)
 - "police" (weight: 0.004)
 - "vote" (weight: 0.004)

Topic 1:
 - "school" (weight: 0.009)
 - "health" (weight: 0.008)
 - "percent" (weight: 0.008)
 - "state" (weight: 0.007)
 - "student" (weight: 0.007)
 - "company" (weight: 0.006)
 - "care" (